# 1. Setup
Import libraries, set seeds, detect device

In [1]:
# %%
"""Setup: Imports, versions, seeds, device detection"""

import os
import gc
import re
import warnings
import hashlib
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

warnings.filterwarnings('ignore')

# Print versions
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# Set seeds for reproducibility
def set_seed(seed: int = 42):
    """Set all random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

# Device detection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {DEVICE}")

NumPy: 2.2.5
Pandas: 2.3.3
PyTorch: 2.9.1+cpu
CUDA available: False

Using device: cpu


# 2. Config
Configuration classes for CPU_DEBUG (fast, laptop-friendly) and GPU_FULL (production quality)

In [2]:
# %%
"""Configuration dataclasses for different execution modes."""

@dataclass
class Config:
    """Configuration for training pipeline."""
    # Mode
    name: str = "CPU_DEBUG"
    
    # Data
    data_dir: str = "data"
    sample_frac: float = 1.0  # Fraction of training data to use
    max_train_rows: Optional[int] = None  # Hard limit on rows
    
    # Features
    title_svd_dim: int = 50  # TF-IDF SVD dimensions for titles
    tag_svd_dim: int = 30   # TF-IDF SVD dimensions for tags
    use_tags: bool = True
    use_title: bool = True
    use_genres: bool = True
    
    # Model architecture
    embed_dim: int = 64     # Embedding dimension for MF
    hidden_dims: List[int] = field(default_factory=lambda: [128, 64])
    dropout: float = 0.2
    
    # Training
    batch_size: int = 4096
    epochs: int = 10
    lr: float = 0.001
    weight_decay: float = 1e-5
    patience: int = 3       # Early stopping patience
    
    # Implicit feedback
    use_implicit: bool = True
    implicit_weight: float = 0.1  # Weight for implicit loss
    
    # Hardware
    num_workers: int = 0
    pin_memory: bool = False
    mixed_precision: bool = False
    
    # Validation
    val_size: float = 0.1
    
    # Predictions
    clip_predictions: bool = True
    round_to_half: bool = False  # Round to nearest 0.5
    
    # Seed
    seed: int = 42


# CPU_DEBUG: Fast execution for testing on laptop
CPU_DEBUG = Config(
    name="CPU_DEBUG",
    sample_frac=0.05,        # Use only 5% of data
    max_train_rows=500_000,  # Max 500k rows
    embed_dim=32,
    hidden_dims=[64, 32],
    title_svd_dim=20,
    tag_svd_dim=15,
    batch_size=2048,
    epochs=3,
    lr=0.005,
    patience=2,
    num_workers=0,
    pin_memory=False,
    mixed_precision=False,
)

# GPU_FULL: Full training for best quality
GPU_FULL = Config(
    name="GPU_FULL",
    sample_frac=1.0,
    max_train_rows=None,
    embed_dim=128,
    hidden_dims=[256, 128, 64],
    title_svd_dim=100,
    tag_svd_dim=50,
    batch_size=8192,
    epochs=30,
    lr=0.001,
    patience=5,
    num_workers=4,
    pin_memory=True,
    mixed_precision=True,
)

# ============================================
# SELECT CONFIGURATION HERE
# ============================================
CFG = GPU_FULL  # Change to GPU_FULL for production
# ============================================

print(f"Using configuration: {CFG.name}")
print(f"  Sample fraction: {CFG.sample_frac}")
print(f"  Embedding dim: {CFG.embed_dim}")
print(f"  Batch size: {CFG.batch_size}")
print(f"  Epochs: {CFG.epochs}")

Using configuration: GPU_FULL
  Sample fraction: 1.0
  Embedding dim: 128
  Batch size: 8192
  Epochs: 30


# 3. Load Data
Efficient data loading with proper dtypes for memory optimization

In [3]:
# %%
"""Load all data files with memory-efficient dtypes."""

def load_data(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load all CSV files with optimized dtypes.
    
    Returns:
        train_df: Training data with user_id, movie_id, rating (can be null)
        movies_df: Movie metadata with movie_id, title, genres
        tags_df: User tags with user_id, movie_id, tag
        submission_df: Submission template with id column
    """
    data_dir = cfg.data_dir
    
    # Load movies
    movies_df = pd.read_csv(
        os.path.join(data_dir, 'movies.csv'),
        dtype={'movie_id': 'int32', 'title': 'str', 'genres': 'str'}
    )
    print(f"Movies: {len(movies_df):,} rows")
    
    # Load tags
    tags_df = pd.read_csv(
        os.path.join(data_dir, 'tags.csv'),
        dtype={'user_id': 'int32', 'movie_id': 'int32', 'tag': 'str'}
    )
    print(f"Tags: {len(tags_df):,} rows")
    
    # Load training data (rating can be null for implicit feedback)
    train_df = pd.read_csv(
        os.path.join(data_dir, 'train.csv'),
        dtype={'user_id': 'int32', 'movie_id': 'int32', 'rating': 'float32'}
    )
    print(f"Train: {len(train_df):,} rows")
    
    # Count explicit vs implicit
    n_explicit = train_df['rating'].notna().sum()
    n_implicit = train_df['rating'].isna().sum()
    print(f"  Explicit ratings: {n_explicit:,} ({100*n_explicit/len(train_df):.1f}%)")
    print(f"  Implicit feedback: {n_implicit:,} ({100*n_implicit/len(train_df):.1f}%)")
    
    # Load submission template
    submission_df = pd.read_csv(
        os.path.join(data_dir, 'ratings_submission.csv'),
        dtype={'id': 'str', 'prediction': 'float32'}
    )
    print(f"Submission: {len(submission_df):,} rows to predict")
    
    return train_df, movies_df, tags_df, submission_df

# Load all data
train_df, movies_df, tags_df, submission_df = load_data(CFG)

# Show samples
print("\n--- Sample Data ---")
print("\nMovies sample:")
print(movies_df.head(3))
print("\nTrain sample (with explicit ratings):")
print(train_df[train_df['rating'].notna()].head(3))
print("\nTrain sample (implicit feedback - null ratings):")
print(train_df[train_df['rating'].isna()].head(3))

Movies: 2,000 rows
Tags: 961,666 rows
Train: 18,321,319 rows
  Explicit ratings: 7,303,350 (39.9%)
  Implicit feedback: 11,017,969 (60.1%)
Submission: 100,000 rows to predict

--- Sample Data ---

Movies sample:
   movie_id                    title  \
0        11         Toy Story (1995)   
1         8           Jumanji (1995)   
2         9  Grumpier Old Men (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  

Train sample (with explicit ratings):
   user_id  movie_id  rating
4        0        31     2.0
5        0        37     4.0
6        0        38     3.0

Train sample (implicit feedback - null ratings):
   user_id  movie_id  rating
0        0         0     NaN
1        0         8     NaN
2        0        11     NaN


# 4. Preprocessing
Build ID mappings, parse submission IDs, create feature matrices

In [4]:
# %%
"""Build ID mappings for users and movies to contiguous indices."""

class IDMapper:
    """Maps original IDs to contiguous indices and back."""
    
    def __init__(self):
        self.user_to_idx: Dict[int, int] = {}
        self.idx_to_user: Dict[int, int] = {}
        self.movie_to_idx: Dict[int, int] = {}
        self.idx_to_movie: Dict[int, int] = {}
        self.n_users: int = 0
        self.n_movies: int = 0
        
    def fit(self, train_df: pd.DataFrame, movies_df: pd.DataFrame, 
            submission_df: pd.DataFrame) -> 'IDMapper':
        """
        Build mappings from all data sources to ensure coverage.
        """
        # Collect all user IDs from train and submission
        train_users = set(train_df['user_id'].unique())
        
        # Parse submission IDs to get user_id and movie_id
        sub_users = set()
        sub_movies = set()
        for id_str in submission_df['id']:
            parts = id_str.split('_')
            sub_users.add(int(parts[0]))
            sub_movies.add(int(parts[1]))
        
        all_users = train_users | sub_users
        all_movies = set(train_df['movie_id'].unique()) | set(movies_df['movie_id'].unique()) | sub_movies
        
        # Create mappings
        for idx, user_id in enumerate(sorted(all_users)):
            self.user_to_idx[user_id] = idx
            self.idx_to_user[idx] = user_id
        
        for idx, movie_id in enumerate(sorted(all_movies)):
            self.movie_to_idx[movie_id] = idx
            self.idx_to_movie[idx] = movie_id
        
        self.n_users = len(self.user_to_idx)
        self.n_movies = len(self.movie_to_idx)
        
        print(f"ID Mapper: {self.n_users:,} users, {self.n_movies:,} movies")
        return self
    
    def transform_users(self, user_ids: np.ndarray) -> np.ndarray:
        """Convert user IDs to indices."""
        return np.array([self.user_to_idx.get(u, -1) for u in user_ids], dtype=np.int32)
    
    def transform_movies(self, movie_ids: np.ndarray) -> np.ndarray:
        """Convert movie IDs to indices."""
        return np.array([self.movie_to_idx.get(m, -1) for m in movie_ids], dtype=np.int32)

# Build ID mapper
id_mapper = IDMapper().fit(train_df, movies_df, submission_df)

# Parse submission IDs
def parse_submission_ids(submission_df: pd.DataFrame) -> pd.DataFrame:
    """Parse 'user_movie' format submission IDs."""
    ids = submission_df['id'].str.split('_', expand=True)
    submission_df = submission_df.copy()
    submission_df['user_id'] = ids[0].astype('int32')
    submission_df['movie_id'] = ids[1].astype('int32')
    return submission_df

submission_df = parse_submission_ids(submission_df)
print(f"\nSubmission IDs parsed: {len(submission_df):,} pairs")

ID Mapper: 100,000 users, 2,000 movies

Submission IDs parsed: 100,000 pairs


In [5]:
# %%
"""Build content feature matrices: genres, titles, tags."""

class FeatureBuilder:
    """Builds and stores all content features for movies."""
    
    def __init__(self, cfg: Config, id_mapper: IDMapper):
        self.cfg = cfg
        self.id_mapper = id_mapper
        self.genre_encoder: Optional[MultiLabelBinarizer] = None
        self.title_vectorizer: Optional[TfidfVectorizer] = None
        self.title_svd: Optional[TruncatedSVD] = None
        self.tag_vectorizer: Optional[TfidfVectorizer] = None
        self.tag_svd: Optional[TruncatedSVD] = None
        
        # Feature matrices (indexed by movie idx)
        self.genre_features: Optional[np.ndarray] = None  # Dense
        self.title_features: Optional[np.ndarray] = None  # Dense (after SVD)
        self.tag_features: Optional[np.ndarray] = None    # Dense (after SVD)
        self.all_features: Optional[np.ndarray] = None    # Combined
        
        self.n_genre_features: int = 0
        self.n_title_features: int = 0
        self.n_tag_features: int = 0
        self.n_total_features: int = 0
        
    def _build_genre_features(self, movies_df: pd.DataFrame) -> np.ndarray:
        """Build multi-hot genre encoding."""
        # Parse genres
        genres_list = movies_df['genres'].fillna('').str.split('|').tolist()
        
        # Fit multi-label binarizer
        self.genre_encoder = MultiLabelBinarizer()
        genre_matrix = self.genre_encoder.fit_transform(genres_list)
        
        # Create full matrix for all movie indices
        n_movies = self.id_mapper.n_movies
        full_genre_matrix = np.zeros((n_movies, genre_matrix.shape[1]), dtype=np.float32)
        
        for i, movie_id in enumerate(movies_df['movie_id']):
            idx = self.id_mapper.movie_to_idx.get(movie_id, -1)
            if idx >= 0:
                full_genre_matrix[idx] = genre_matrix[i]
        
        self.n_genre_features = full_genre_matrix.shape[1]
        print(f"Genre features: {self.n_genre_features} dimensions")
        return full_genre_matrix
    
    def _build_title_features(self, movies_df: pd.DataFrame) -> np.ndarray:
        """Build TF-IDF + SVD features from movie titles."""
        # Extract year and clean title
        titles = movies_df['title'].fillna('').tolist()
        
        # TF-IDF vectorization
        self.title_vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words='english',
            min_df=2
        )
        title_tfidf = self.title_vectorizer.fit_transform(titles)
        
        # Dimensionality reduction with SVD
        n_components = min(self.cfg.title_svd_dim, title_tfidf.shape[1] - 1)
        self.title_svd = TruncatedSVD(n_components=n_components, random_state=42)
        title_svd_features = self.title_svd.fit_transform(title_tfidf)
        
        # Create full matrix
        n_movies = self.id_mapper.n_movies
        full_title_matrix = np.zeros((n_movies, n_components), dtype=np.float32)
        
        for i, movie_id in enumerate(movies_df['movie_id']):
            idx = self.id_mapper.movie_to_idx.get(movie_id, -1)
            if idx >= 0:
                full_title_matrix[idx] = title_svd_features[i]
        
        self.n_title_features = n_components
        print(f"Title features: {self.n_title_features} dimensions (SVD from TF-IDF)")
        return full_title_matrix
    
    def _build_tag_features(self, tags_df: pd.DataFrame) -> np.ndarray:
        """Build TF-IDF + SVD features from user tags aggregated per movie."""
        # Aggregate tags per movie
        movie_tags = tags_df.groupby('movie_id')['tag'].apply(
            lambda x: ' '.join(x.dropna().astype(str))
        ).reset_index()
        
        # TF-IDF vectorization
        self.tag_vectorizer = TfidfVectorizer(
            max_features=3000,
            ngram_range=(1, 2),
            min_df=2
        )
        
        tag_texts = movie_tags['tag'].tolist()
        if len(tag_texts) == 0:
            print("No tags available, skipping tag features")
            return np.zeros((self.id_mapper.n_movies, 1), dtype=np.float32)
        
        tag_tfidf = self.tag_vectorizer.fit_transform(tag_texts)
        
        # Dimensionality reduction
        n_components = min(self.cfg.tag_svd_dim, tag_tfidf.shape[1] - 1, len(tag_texts) - 1)
        if n_components < 1:
            n_components = 1
        self.tag_svd = TruncatedSVD(n_components=n_components, random_state=42)
        tag_svd_features = self.tag_svd.fit_transform(tag_tfidf)
        
        # Create full matrix
        n_movies = self.id_mapper.n_movies
        full_tag_matrix = np.zeros((n_movies, n_components), dtype=np.float32)
        
        for i, movie_id in enumerate(movie_tags['movie_id']):
            idx = self.id_mapper.movie_to_idx.get(movie_id, -1)
            if idx >= 0:
                full_tag_matrix[idx] = tag_svd_features[i]
        
        self.n_tag_features = n_components
        print(f"Tag features: {self.n_tag_features} dimensions (SVD from TF-IDF)")
        return full_tag_matrix
    
    def build(self, movies_df: pd.DataFrame, tags_df: pd.DataFrame) -> 'FeatureBuilder':
        """Build all feature matrices."""
        print("\nBuilding content features...")
        
        features_list = []
        
        if self.cfg.use_genres:
            self.genre_features = self._build_genre_features(movies_df)
            features_list.append(self.genre_features)
        
        if self.cfg.use_title:
            self.title_features = self._build_title_features(movies_df)
            features_list.append(self.title_features)
        
        if self.cfg.use_tags and len(tags_df) > 0:
            self.tag_features = self._build_tag_features(tags_df)
            features_list.append(self.tag_features)
        
        # Combine all features
        if features_list:
            self.all_features = np.hstack(features_list).astype(np.float32)
            self.n_total_features = self.all_features.shape[1]
            print(f"\nTotal content features: {self.n_total_features} dimensions")
        else:
            self.all_features = np.zeros((self.id_mapper.n_movies, 1), dtype=np.float32)
            self.n_total_features = 1
        
        return self
    
    def get_movie_features(self, movie_indices: np.ndarray) -> np.ndarray:
        """Get content features for given movie indices."""
        return self.all_features[movie_indices]

# Build features
feature_builder = FeatureBuilder(CFG, id_mapper).build(movies_df, tags_df)


Building content features...
Genre features: 20 dimensions
Title features: 100 dimensions (SVD from TF-IDF)
Tag features: 50 dimensions (SVD from TF-IDF)

Total content features: 170 dimensions


# 5. Train/Validation Split
User-aware split ensuring each user appears in training set

In [6]:
# %%
"""Train/validation split with sampling and user-aware splitting."""

def prepare_train_data(
    train_df: pd.DataFrame,
    cfg: Config,
    id_mapper: IDMapper
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Prepare training data with sampling and train/val split.
    
    Returns:
        train_explicit: Explicit ratings for training
        val_explicit: Explicit ratings for validation
        train_implicit: Implicit feedback for training
        val_implicit: Implicit feedback for validation
    """
    set_seed(cfg.seed)
    
    # Separate explicit and implicit
    explicit_df = train_df[train_df['rating'].notna()].copy()
    implicit_df = train_df[train_df['rating'].isna()].copy()
    
    print(f"Original explicit ratings: {len(explicit_df):,}")
    print(f"Original implicit feedback: {len(implicit_df):,}")
    
    # Sample if needed (for CPU_DEBUG mode)
    if cfg.sample_frac < 1.0:
        explicit_df = explicit_df.sample(frac=cfg.sample_frac, random_state=cfg.seed)
        implicit_df = implicit_df.sample(frac=cfg.sample_frac, random_state=cfg.seed)
        print(f"\nAfter sampling ({cfg.sample_frac:.1%}):")
        print(f"  Explicit: {len(explicit_df):,}")
        print(f"  Implicit: {len(implicit_df):,}")
    
    if cfg.max_train_rows is not None:
        if len(explicit_df) > cfg.max_train_rows:
            explicit_df = explicit_df.head(cfg.max_train_rows)
        if len(implicit_df) > cfg.max_train_rows:
            implicit_df = implicit_df.head(cfg.max_train_rows)
        print(f"\nAfter max_rows limit ({cfg.max_train_rows:,}):")
        print(f"  Explicit: {len(explicit_df):,}")
        print(f"  Implicit: {len(implicit_df):,}")
    
    # Convert IDs to indices
    explicit_df['user_idx'] = id_mapper.transform_users(explicit_df['user_id'].values)
    explicit_df['movie_idx'] = id_mapper.transform_movies(explicit_df['movie_id'].values)
    implicit_df['user_idx'] = id_mapper.transform_users(implicit_df['user_id'].values)
    implicit_df['movie_idx'] = id_mapper.transform_movies(implicit_df['movie_id'].values)
    
    # User-aware train/val split for explicit ratings
    # Ensure each user has at least one rating in training
    def user_aware_split(df: pd.DataFrame, val_size: float, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Split data ensuring each user appears in training."""
        # Group by user and keep at least one per user in train
        train_indices = []
        val_indices = []
        
        for user_id, group in df.groupby('user_id'):
            indices = group.index.tolist()
            if len(indices) == 1:
                # Only one rating - keep in train
                train_indices.extend(indices)
            else:
                # Split user's ratings
                n_val = max(1, int(len(indices) * val_size))
                np.random.shuffle(indices)
                val_indices.extend(indices[:n_val])
                train_indices.extend(indices[n_val:])
        
        return df.loc[train_indices].copy(), df.loc[val_indices].copy()
    
    train_explicit, val_explicit = user_aware_split(explicit_df, cfg.val_size, cfg.seed)
    
    # For implicit, use simple random split
    train_implicit, val_implicit = train_test_split(
        implicit_df, test_size=cfg.val_size, random_state=cfg.seed
    )
    
    print(f"\nFinal split:")
    print(f"  Train explicit: {len(train_explicit):,}")
    print(f"  Val explicit: {len(val_explicit):,}")
    print(f"  Train implicit: {len(train_implicit):,}")
    print(f"  Val implicit: {len(val_implicit):,}")
    
    return train_explicit, val_explicit, train_implicit, val_implicit

# Prepare data
train_explicit, val_explicit, train_implicit, val_implicit = prepare_train_data(
    train_df, CFG, id_mapper
)

# Compute global statistics from training set
GLOBAL_MEAN = train_explicit['rating'].mean()
print(f"\nGlobal mean rating: {GLOBAL_MEAN:.4f}")

Original explicit ratings: 7,303,350
Original implicit feedback: 11,017,969

Final split:
  Train explicit: 6,615,426
  Val explicit: 687,924
  Train implicit: 9,916,172
  Val implicit: 1,101,797

Global mean rating: 3.6081


# 6. Metric: Weighted RMSE
Implementation of W-RMSE where movies with many ratings are down-weighted

In [7]:
# %%
"""Weighted RMSE metric implementation."""

def compute_movie_weights(
    train_explicit: pd.DataFrame,
    movie_indices: np.ndarray,
    alpha: float = 0.5,
    min_count: int = 1
) -> np.ndarray:
    """
    Compute weights for each movie based on rating counts.
    
    Movies with more ratings get lower weights (down-weighted).
    
    Formula: weight_i = 1 / (count_i ^ alpha)
    
    This ensures movies with many ratings don't dominate the metric.
    Normalized so weights sum to len(movie_indices).
    
    Args:
        train_explicit: Training data with explicit ratings
        movie_indices: Array of movie indices for which to compute weights
        alpha: Exponent for down-weighting (0.5 = sqrt, 1 = linear)
        min_count: Minimum count to use (smoothing)
    
    Returns:
        Normalized weights array
    """
    # Count ratings per movie in training set
    movie_counts = train_explicit.groupby('movie_idx').size()
    
    # Get counts for given movies (use min_count for unseen movies)
    counts = np.array([
        movie_counts.get(idx, min_count) for idx in movie_indices
    ], dtype=np.float32)
    
    # Compute inverse frequency weights
    weights = 1.0 / np.power(np.maximum(counts, min_count), alpha)
    
    # Normalize so weights sum to n
    weights = weights * len(weights) / weights.sum()
    
    return weights


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute standard RMSE."""
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def wrmse(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    movie_indices: np.ndarray,
    train_explicit: pd.DataFrame,
    alpha: float = 0.5
) -> float:
    """
    Compute Weighted RMSE.
    
    Movies with many ratings are down-weighted to prevent them from
    dominating the metric.
    
    Args:
        y_true: True ratings
        y_pred: Predicted ratings
        movie_indices: Movie indices for each prediction
        train_explicit: Training explicit data (for computing counts)
        alpha: Down-weighting exponent (default 0.5)
    
    Returns:
        Weighted RMSE value
    
    Note:
        Adjust alpha if needed based on competition specifics.
        alpha=0.5 is a common choice (inverse sqrt of count).
    """
    weights = compute_movie_weights(train_explicit, movie_indices, alpha=alpha)
    
    squared_errors = (y_true - y_pred) ** 2
    weighted_mse = np.sum(weights * squared_errors) / np.sum(weights)
    
    return np.sqrt(weighted_mse)


# Pre-compute movie counts for efficient weight computation
movie_rating_counts = train_explicit.groupby('movie_idx').size().to_dict()

print("W-RMSE metric functions defined.")
print(f"Training set has {len(movie_rating_counts)} unique movies with explicit ratings.")

W-RMSE metric functions defined.
Training set has 2000 unique movies with explicit ratings.


# 7. Baselines
Global mean, movie mean with shrinkage, and user+movie bias model

In [ ]:
# %%
"""Baseline models for comparison."""

class GlobalMeanBaseline:
    """Predict global mean for all ratings."""
    
    def __init__(self):
        self.global_mean: float = 0.0
        
    def fit(self, train_explicit: pd.DataFrame) -> 'GlobalMeanBaseline':
        self.global_mean = train_explicit['rating'].mean()
        return self
    
    def predict(self, user_indices: np.ndarray, movie_indices: np.ndarray) -> np.ndarray:
        return np.full(len(user_indices), self.global_mean, dtype=np.float32)


class MovieMeanBaseline:
    """Predict movie mean with shrinkage towards global mean."""
    
    def __init__(self, shrinkage: float = 25.0):
        self.shrinkage = shrinkage
        self.global_mean: float = 0.0
        self.movie_means: Dict[int, float] = {}
        
    def fit(self, train_explicit: pd.DataFrame) -> 'MovieMeanBaseline':
        self.global_mean = train_explicit['rating'].mean()
        
        # Compute shrunk movie means
        movie_stats = train_explicit.groupby('movie_idx')['rating'].agg(['sum', 'count'])
        for movie_idx, row in movie_stats.iterrows():
            shrunk_mean = (row['sum'] + self.shrinkage * self.global_mean) / (row['count'] + self.shrinkage)
            self.movie_means[movie_idx] = shrunk_mean
        
        return self
    
    def predict(self, user_indices: np.ndarray, movie_indices: np.ndarray) -> np.ndarray:
        preds = np.array([
            self.movie_means.get(m, self.global_mean) for m in movie_indices
        ], dtype=np.float32)
        return preds


class BiasBaseline:
    """User + Movie bias model: pred = global_mean + user_bias + movie_bias"""
    
    def __init__(self, shrinkage_user: float = 25.0, shrinkage_movie: float = 25.0):
        self.shrinkage_user = shrinkage_user
        self.shrinkage_movie = shrinkage_movie
        self.global_mean: float = 0.0
        self.user_biases: Dict[int, float] = {}
        self.movie_biases: Dict[int, float] = {}
        
    def fit(self, train_explicit: pd.DataFrame, n_iterations: int = 10) -> 'BiasBaseline':
        """Fit biases using alternating least squares."""
        self.global_mean = train_explicit['rating'].mean()
        
        # Initialize biases to zero
        users = train_explicit['user_idx'].unique()
        movies = train_explicit['movie_idx'].unique()
        
        user_biases = {u: 0.0 for u in users}
        movie_biases = {m: 0.0 for m in movies}
        
        # Alternating optimization
        for iteration in range(n_iterations):
            # Update movie biases
            for movie_idx in movies:
                mask = train_explicit['movie_idx'] == movie_idx
                if mask.sum() == 0:
                    continue
                subset = train_explicit[mask]
                residuals = subset['rating'].values - self.global_mean - np.array([
                    user_biases.get(u, 0) for u in subset['user_idx'].values
                ])
                movie_biases[movie_idx] = residuals.sum() / (len(residuals) + self.shrinkage_movie)
            
            # Update user biases
            for user_idx in users:
                mask = train_explicit['user_idx'] == user_idx
                if mask.sum() == 0:
                    continue
                subset = train_explicit[mask]
                residuals = subset['rating'].values - self.global_mean - np.array([
                    movie_biases.get(m, 0) for m in subset['movie_idx'].values
                ])
                user_biases[user_idx] = residuals.sum() / (len(residuals) + self.shrinkage_user)
        
        self.user_biases = user_biases
        self.movie_biases = movie_biases
        return self
    
    def predict(self, user_indices: np.ndarray, movie_indices: np.ndarray) -> np.ndarray:
        preds = np.array([
            self.global_mean + self.user_biases.get(u, 0) + self.movie_biases.get(m, 0)
            for u, m in zip(user_indices, movie_indices)
        ], dtype=np.float32)
        return np.clip(preds, 0.5, 5.0)


# Evaluate baselines
print("Evaluating baselines on validation set...\n")

val_user_idx = val_explicit['user_idx'].values
val_movie_idx = val_explicit['movie_idx'].values
val_ratings = val_explicit['rating'].values

# Global Mean
global_baseline = GlobalMeanBaseline().fit(train_explicit)
preds_global = global_baseline.predict(val_user_idx, val_movie_idx)
rmse_global = rmse(val_ratings, preds_global)
wrmse_global = wrmse(val_ratings, preds_global, val_movie_idx, train_explicit)
print(f"Global Mean Baseline:")
print(f"  RMSE:  {rmse_global:.4f}")
print(f"  W-RMSE: {wrmse_global:.4f}")

# Movie Mean with Shrinkage
movie_baseline = MovieMeanBaseline(shrinkage=25.0).fit(train_explicit)
preds_movie = movie_baseline.predict(val_user_idx, val_movie_idx)
rmse_movie = rmse(val_ratings, preds_movie)
wrmse_movie = wrmse(val_ratings, preds_movie, val_movie_idx, train_explicit)
print(f"\nMovie Mean Baseline (shrinkage=25):")
print(f"  RMSE:  {rmse_movie:.4f}")
print(f"  W-RMSE: {wrmse_movie:.4f}")

# User + Movie Bias
bias_baseline = BiasBaseline(shrinkage_user=25, shrinkage_movie=25).fit(train_explicit)
preds_bias = bias_baseline.predict(val_user_idx, val_movie_idx)
rmse_bias = rmse(val_ratings, preds_bias)
wrmse_bias = wrmse(val_ratings, preds_bias, val_movie_idx, train_explicit)
print(f"\nUser+Movie Bias Baseline:")
print(f"  RMSE:  {rmse_bias:.4f}")
print(f"  W-RMSE: {wrmse_bias:.4f}")

# Store best baseline for comparison
best_baseline_wrmse = min(wrmse_global, wrmse_movie, wrmse_bias)
print(f"\nBest baseline W-RMSE: {best_baseline_wrmse:.4f}")

Evaluating baselines on validation set...

Global Mean Baseline:
  RMSE:  1.0226
  W-RMSE: 1.0353

Movie Mean Baseline (shrinkage=25):
  RMSE:  0.9400
  W-RMSE: 0.9460


# 8. Candidate Models
PyTorch implementations of Biased MF and Hybrid MF + Content models

In [ ]:
# %%
"""PyTorch Dataset for rating prediction."""

class RatingDataset(Dataset):
    """Dataset for explicit ratings."""
    
    def __init__(
        self,
        user_indices: np.ndarray,
        movie_indices: np.ndarray,
        ratings: np.ndarray,
        movie_features: Optional[np.ndarray] = None
    ):
        self.user_indices = torch.LongTensor(user_indices)
        self.movie_indices = torch.LongTensor(movie_indices)
        self.ratings = torch.FloatTensor(ratings)
        self.movie_features = (
            torch.FloatTensor(movie_features) if movie_features is not None else None
        )
        
    def __len__(self) -> int:
        return len(self.ratings)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, ...]:
        if self.movie_features is not None:
            return (
                self.user_indices[idx],
                self.movie_indices[idx],
                self.ratings[idx],
                self.movie_features[idx]
            )
        return (
            self.user_indices[idx],
            self.movie_indices[idx],
            self.ratings[idx]
        )


class ImplicitDataset(Dataset):
    """Dataset for implicit feedback (watched but not rated)."""
    
    def __init__(
        self,
        user_indices: np.ndarray,
        movie_indices: np.ndarray,
        movie_features: Optional[np.ndarray] = None
    ):
        self.user_indices = torch.LongTensor(user_indices)
        self.movie_indices = torch.LongTensor(movie_indices)
        self.movie_features = (
            torch.FloatTensor(movie_features) if movie_features is not None else None
        )
        # Target is 1.0 (positive interaction)
        self.targets = torch.ones(len(user_indices), dtype=torch.float32)
        
    def __len__(self) -> int:
        return len(self.user_indices)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, ...]:
        if self.movie_features is not None:
            return (
                self.user_indices[idx],
                self.movie_indices[idx],
                self.targets[idx],
                self.movie_features[idx]
            )
        return (
            self.user_indices[idx],
            self.movie_indices[idx],
            self.targets[idx]
        )

print("Dataset classes defined.")

Dataset classes defined.


In [ ]:
# %%
"""Candidate A: Biased Matrix Factorization Model."""

class BiasedMF(nn.Module):
    """
    Biased Matrix Factorization for explicit ratings.
    
    Prediction: global_mean + user_bias + item_bias + user_embed · item_embed
    """
    
    def __init__(
        self,
        n_users: int,
        n_movies: int,
        embed_dim: int = 64,
        global_mean: float = 3.5,
        dropout: float = 0.0
    ):
        super().__init__()
        
        self.global_mean = nn.Parameter(torch.tensor(global_mean), requires_grad=False)
        
        # Embeddings
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.movie_embed = nn.Embedding(n_movies, embed_dim)
        
        # Biases
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        
        self.dropout = nn.Dropout(dropout)
        
        # Initialize
        self._init_weights()
        
    def _init_weights(self):
        """Initialize embeddings with small random values."""
        nn.init.normal_(self.user_embed.weight, std=0.01)
        nn.init.normal_(self.movie_embed.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)
        
    def forward(
        self,
        user_idx: torch.Tensor,
        movie_idx: torch.Tensor
    ) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            user_idx: User indices (batch_size,)
            movie_idx: Movie indices (batch_size,)
            
        Returns:
            Predicted ratings (batch_size,)
        """
        # Get embeddings
        u_embed = self.dropout(self.user_embed(user_idx))
        m_embed = self.dropout(self.movie_embed(movie_idx))
        
        # Get biases
        u_bias = self.user_bias(user_idx).squeeze(-1)
        m_bias = self.movie_bias(movie_idx).squeeze(-1)
        
        # Compute prediction
        dot_product = (u_embed * m_embed).sum(dim=-1)
        pred = self.global_mean + u_bias + m_bias + dot_product
        
        return pred
    
    def get_user_embeddings(self, user_idx: torch.Tensor) -> torch.Tensor:
        """Get user embeddings for given indices."""
        return self.user_embed(user_idx)
    
    def get_movie_embeddings(self, movie_idx: torch.Tensor) -> torch.Tensor:
        """Get movie embeddings for given indices."""
        return self.movie_embed(movie_idx)

print("BiasedMF model defined.")

BiasedMF model defined.


In [ ]:
# %%
"""Candidate B: Hybrid MF + Content Model."""

class HybridMF(nn.Module):
    """
    Hybrid Matrix Factorization with content features.
    
    Combines collaborative filtering with content-based features:
    - User/movie embeddings and biases (like BiasedMF)
    - Movie content features projected and added
    - Optional MLP for feature interactions
    """
    
    def __init__(
        self,
        n_users: int,
        n_movies: int,
        embed_dim: int = 64,
        content_dim: int = 50,
        hidden_dims: List[int] = [128, 64],
        global_mean: float = 3.5,
        dropout: float = 0.2
    ):
        super().__init__()
        
        self.global_mean = nn.Parameter(torch.tensor(global_mean), requires_grad=False)
        
        # Embeddings
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.movie_embed = nn.Embedding(n_movies, embed_dim)
        
        # Biases
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        
        # Content feature projection
        self.content_proj = nn.Linear(content_dim, embed_dim)
        
        # MLP for combining features
        mlp_input_dim = embed_dim * 3  # user_embed, movie_embed, content_proj
        layers = []
        prev_dim = mlp_input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.mlp = nn.Sequential(*layers)
        
        self.dropout = nn.Dropout(dropout)
        
        # Blend weight between MF and MLP
        self.blend_weight = nn.Parameter(torch.tensor(0.5))
        
        self._init_weights()
        
    def _init_weights(self):
        """Initialize weights."""
        nn.init.normal_(self.user_embed.weight, std=0.01)
        nn.init.normal_(self.movie_embed.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)
        nn.init.xavier_uniform_(self.content_proj.weight)
        
    def forward(
        self,
        user_idx: torch.Tensor,
        movie_idx: torch.Tensor,
        content_features: torch.Tensor
    ) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            user_idx: User indices (batch_size,)
            movie_idx: Movie indices (batch_size,)
            content_features: Movie content features (batch_size, content_dim)
            
        Returns:
            Predicted ratings (batch_size,)
        """
        # Get embeddings
        u_embed = self.dropout(self.user_embed(user_idx))
        m_embed = self.dropout(self.movie_embed(movie_idx))
        
        # Get biases
        u_bias = self.user_bias(user_idx).squeeze(-1)
        m_bias = self.movie_bias(movie_idx).squeeze(-1)
        
        # Project content features
        content_proj = self.dropout(torch.relu(self.content_proj(content_features)))
        
        # MF component
        mf_pred = (u_embed * m_embed).sum(dim=-1)
        
        # MLP component with all features
        mlp_input = torch.cat([u_embed, m_embed, content_proj], dim=-1)
        mlp_pred = self.mlp(mlp_input).squeeze(-1)
        
        # Blend MF and MLP
        blend = torch.sigmoid(self.blend_weight)
        combined = blend * mf_pred + (1 - blend) * mlp_pred
        
        # Final prediction
        pred = self.global_mean + u_bias + m_bias + combined
        
        return pred

print("HybridMF model defined.")

HybridMF model defined.


In [ ]:
# %%
"""Candidate C: Residual Hybrid Model - MF + Content correction."""

class ResidualHybrid(nn.Module):
    """
    Residual model that learns to correct a base MF prediction using content features.
    
    Prediction: MF_prediction + residual_correction(content_features, user_embed, movie_embed)
    """
    
    def __init__(
        self,
        n_users: int,
        n_movies: int,
        embed_dim: int = 64,
        content_dim: int = 50,
        hidden_dims: List[int] = [64, 32],
        global_mean: float = 3.5,
        dropout: float = 0.2
    ):
        super().__init__()
        
        self.global_mean = nn.Parameter(torch.tensor(global_mean), requires_grad=False)
        
        # Base MF components
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.movie_embed = nn.Embedding(n_movies, embed_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        
        # Residual correction network
        # Takes content features + embeddings and outputs a small correction
        residual_input_dim = content_dim + embed_dim * 2
        layers = []
        prev_dim = residual_input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.residual_net = nn.Sequential(*layers)
        
        # Scale for residual (starts small)
        self.residual_scale = nn.Parameter(torch.tensor(0.1))
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
        
    def _init_weights(self):
        nn.init.normal_(self.user_embed.weight, std=0.01)
        nn.init.normal_(self.movie_embed.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)
        
    def forward(
        self,
        user_idx: torch.Tensor,
        movie_idx: torch.Tensor,
        content_features: torch.Tensor
    ) -> torch.Tensor:
        # Get embeddings
        u_embed = self.dropout(self.user_embed(user_idx))
        m_embed = self.dropout(self.movie_embed(movie_idx))
        
        # Get biases
        u_bias = self.user_bias(user_idx).squeeze(-1)
        m_bias = self.movie_bias(movie_idx).squeeze(-1)
        
        # Base MF prediction
        mf_pred = self.global_mean + u_bias + m_bias + (u_embed * m_embed).sum(dim=-1)
        
        # Residual correction
        residual_input = torch.cat([content_features, u_embed, m_embed], dim=-1)
        residual = self.residual_net(residual_input).squeeze(-1)
        
        # Final prediction with scaled residual
        pred = mf_pred + self.residual_scale * residual
        
        return pred

print("ResidualHybrid model defined.")

ResidualHybrid model defined.


In [ ]:
# %%
"""Training utilities: trainer class with early stopping and checkpointing."""

class Trainer:
    """Trainer for recommendation models with early stopping."""
    
    def __init__(
        self,
        model: nn.Module,
        cfg: Config,
        train_explicit: pd.DataFrame,
        val_explicit: pd.DataFrame,
        feature_builder: FeatureBuilder,
        use_content: bool = False,
        model_name: str = "model"
    ):
        self.model = model.to(DEVICE)
        self.cfg = cfg
        self.train_explicit = train_explicit
        self.val_explicit = val_explicit
        self.feature_builder = feature_builder
        self.use_content = use_content
        self.model_name = model_name
        
        # Setup optimizer
        self.optimizer = torch.optim.Adam(
            model.parameters(),
            lr=cfg.lr,
            weight_decay=cfg.weight_decay
        )
        
        # Setup loss
        self.criterion = nn.MSELoss()
        
        # Mixed precision
        self.scaler = torch.amp.GradScaler() if cfg.mixed_precision else None
        
        # Tracking
        self.best_wrmse = float('inf')
        self.best_state = None
        self.patience_counter = 0
        self.history = {'train_loss': [], 'val_rmse': [], 'val_wrmse': []}
        
    def _create_dataloader(self, df: pd.DataFrame, shuffle: bool = True) -> DataLoader:
        """Create DataLoader from dataframe."""
        user_idx = df['user_idx'].values
        movie_idx = df['movie_idx'].values
        ratings = df['rating'].values
        
        if self.use_content:
            content_features = self.feature_builder.get_movie_features(movie_idx)
            dataset = RatingDataset(user_idx, movie_idx, ratings, content_features)
        else:
            dataset = RatingDataset(user_idx, movie_idx, ratings)
        
        return DataLoader(
            dataset,
            batch_size=self.cfg.batch_size,
            shuffle=shuffle,
            num_workers=self.cfg.num_workers,
            pin_memory=self.cfg.pin_memory
        )
    
    def train_epoch(self, dataloader: DataLoader) -> float:
        """Train for one epoch."""
        self.model.train()
        total_loss = 0.0
        n_batches = 0
        
        for batch in dataloader:
            if self.use_content:
                user_idx, movie_idx, ratings, content = batch
                user_idx = user_idx.to(DEVICE)
                movie_idx = movie_idx.to(DEVICE)
                ratings = ratings.to(DEVICE)
                content = content.to(DEVICE)
            else:
                user_idx, movie_idx, ratings = batch
                user_idx = user_idx.to(DEVICE)
                movie_idx = movie_idx.to(DEVICE)
                ratings = ratings.to(DEVICE)
                content = None
            
            self.optimizer.zero_grad()
            
            if self.cfg.mixed_precision:
                with torch.amp.autocast(device_type='cuda'):
                    if content is not None:
                        preds = self.model(user_idx, movie_idx, content)
                    else:
                        preds = self.model(user_idx, movie_idx)
                    loss = self.criterion(preds, ratings)
                
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                if content is not None:
                    preds = self.model(user_idx, movie_idx, content)
                else:
                    preds = self.model(user_idx, movie_idx)
                loss = self.criterion(preds, ratings)
                loss.backward()
                self.optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
        
        return total_loss / n_batches
    
    @torch.no_grad()
    def evaluate(self, dataloader: DataLoader) -> Tuple[float, float, np.ndarray, np.ndarray]:
        """Evaluate on validation set."""
        self.model.eval()
        all_preds = []
        all_ratings = []
        all_movie_idx = []
        
        for batch in dataloader:
            if self.use_content:
                user_idx, movie_idx, ratings, content = batch
                user_idx = user_idx.to(DEVICE)
                movie_idx = movie_idx.to(DEVICE)
                content = content.to(DEVICE)
            else:
                user_idx, movie_idx, ratings = batch
                user_idx = user_idx.to(DEVICE)
                movie_idx = movie_idx.to(DEVICE)
                content = None
            
            if content is not None:
                preds = self.model(user_idx, movie_idx, content)
            else:
                preds = self.model(user_idx, movie_idx)
            
            all_preds.extend(preds.cpu().numpy())
            all_ratings.extend(ratings.numpy())
            all_movie_idx.extend(movie_idx.cpu().numpy())
        
        all_preds = np.array(all_preds, dtype=np.float32)
        all_ratings = np.array(all_ratings, dtype=np.float32)
        all_movie_idx = np.array(all_movie_idx, dtype=np.int32)
        
        # Clip predictions
        if self.cfg.clip_predictions:
            all_preds = np.clip(all_preds, 0.5, 5.0)
        
        val_rmse = rmse(all_ratings, all_preds)
        val_wrmse = wrmse(all_ratings, all_preds, all_movie_idx, self.train_explicit)
        
        return val_rmse, val_wrmse, all_preds, all_ratings
    
    def train(self) -> Dict[str, Any]:
        """Full training loop with early stopping."""
        train_loader = self._create_dataloader(self.train_explicit, shuffle=True)
        val_loader = self._create_dataloader(self.val_explicit, shuffle=False)
        
        print(f"\nTraining {self.model_name}...")
        print(f"  Train samples: {len(self.train_explicit):,}")
        print(f"  Val samples: {len(self.val_explicit):,}")
        print(f"  Epochs: {self.cfg.epochs}, Batch size: {self.cfg.batch_size}")
        
        for epoch in range(self.cfg.epochs):
            train_loss = self.train_epoch(train_loader)
            val_rmse, val_wrmse, _, _ = self.evaluate(val_loader)
            
            self.history['train_loss'].append(train_loss)
            self.history['val_rmse'].append(val_rmse)
            self.history['val_wrmse'].append(val_wrmse)
            
            # Early stopping check
            if val_wrmse < self.best_wrmse:
                self.best_wrmse = val_wrmse
                self.best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                self.patience_counter = 0
                improved = " *"
            else:
                self.patience_counter += 1
                improved = ""
            
            print(f"  Epoch {epoch+1}/{self.cfg.epochs}: "
                  f"Loss={train_loss:.4f}, RMSE={val_rmse:.4f}, W-RMSE={val_wrmse:.4f}{improved}")
            
            if self.patience_counter >= self.cfg.patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break
        
        # Restore best model
        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)
            self.model.to(DEVICE)
        
        print(f"  Best W-RMSE: {self.best_wrmse:.4f}")
        
        return {
            'model_name': self.model_name,
            'best_wrmse': self.best_wrmse,
            'history': self.history
        }

print("Trainer class defined.")

Trainer class defined.


# 9. Model Selection
Train all candidate models and select the best by W-RMSE

In [ ]:
# %%
"""Model selection: train all candidates and pick the best."""

def run_model_selection(
    cfg: Config,
    id_mapper: IDMapper,
    feature_builder: FeatureBuilder,
    train_explicit: pd.DataFrame,
    val_explicit: pd.DataFrame,
    global_mean: float
) -> Tuple[str, nn.Module, Dict[str, Any]]:
    """
    Train all candidate models and select the best by W-RMSE.
    
    Returns:
        best_model_name: Name of the best model
        best_model: The trained model
        results: Dictionary of all results
    """
    results = {}
    
    # ==========================================
    # Candidate A: Biased Matrix Factorization
    # ==========================================
    print("\n" + "="*60)
    print("Candidate A: Biased Matrix Factorization")
    print("="*60)
    
    model_a = BiasedMF(
        n_users=id_mapper.n_users,
        n_movies=id_mapper.n_movies,
        embed_dim=cfg.embed_dim,
        global_mean=global_mean,
        dropout=cfg.dropout
    )
    
    trainer_a = Trainer(
        model=model_a,
        cfg=cfg,
        train_explicit=train_explicit,
        val_explicit=val_explicit,
        feature_builder=feature_builder,
        use_content=False,
        model_name="BiasedMF"
    )
    
    results['BiasedMF'] = trainer_a.train()
    results['BiasedMF']['model'] = model_a
    
    # ==========================================
    # Candidate B: Hybrid MF + Content
    # ==========================================
    print("\n" + "="*60)
    print("Candidate B: Hybrid MF + Content")
    print("="*60)
    
    model_b = HybridMF(
        n_users=id_mapper.n_users,
        n_movies=id_mapper.n_movies,
        embed_dim=cfg.embed_dim,
        content_dim=feature_builder.n_total_features,
        hidden_dims=cfg.hidden_dims,
        global_mean=global_mean,
        dropout=cfg.dropout
    )
    
    trainer_b = Trainer(
        model=model_b,
        cfg=cfg,
        train_explicit=train_explicit,
        val_explicit=val_explicit,
        feature_builder=feature_builder,
        use_content=True,
        model_name="HybridMF"
    )
    
    results['HybridMF'] = trainer_b.train()
    results['HybridMF']['model'] = model_b
    
    # ==========================================
    # Candidate C: Residual Hybrid
    # ==========================================
    print("\n" + "="*60)
    print("Candidate C: Residual Hybrid")
    print("="*60)
    
    model_c = ResidualHybrid(
        n_users=id_mapper.n_users,
        n_movies=id_mapper.n_movies,
        embed_dim=cfg.embed_dim,
        content_dim=feature_builder.n_total_features,
        hidden_dims=[d // 2 for d in cfg.hidden_dims],  # Smaller for residual
        global_mean=global_mean,
        dropout=cfg.dropout
    )
    
    trainer_c = Trainer(
        model=model_c,
        cfg=cfg,
        train_explicit=train_explicit,
        val_explicit=val_explicit,
        feature_builder=feature_builder,
        use_content=True,
        model_name="ResidualHybrid"
    )
    
    results['ResidualHybrid'] = trainer_c.train()
    results['ResidualHybrid']['model'] = model_c
    
    # ==========================================
    # Select best model
    # ==========================================
    print("\n" + "="*60)
    print("Model Selection Summary")
    print("="*60)
    
    for name, res in results.items():
        print(f"  {name}: W-RMSE = {res['best_wrmse']:.4f}")
    
    best_model_name = min(results, key=lambda x: results[x]['best_wrmse'])
    best_model = results[best_model_name]['model']
    best_wrmse = results[best_model_name]['best_wrmse']
    
    print(f"\n  *** Best Model: {best_model_name} (W-RMSE = {best_wrmse:.4f}) ***")
    
    return best_model_name, best_model, results

# Run model selection
set_seed(CFG.seed)
best_model_name, best_model, selection_results = run_model_selection(
    cfg=CFG,
    id_mapper=id_mapper,
    feature_builder=feature_builder,
    train_explicit=train_explicit,
    val_explicit=val_explicit,
    global_mean=GLOBAL_MEAN
)


Candidate A: Biased Matrix Factorization

Training BiasedMF...
  Train samples: 296,113
  Val samples: 69,055
  Epochs: 3, Batch size: 2048
  Epoch 1/3: Loss=0.9637, RMSE=0.9570, W-RMSE=0.9697 *
  Epoch 2/3: Loss=0.7908, RMSE=0.9362, W-RMSE=0.9457 *
  Epoch 3/3: Loss=0.5308, RMSE=0.9395, W-RMSE=0.9448 *
  Best W-RMSE: 0.9448

Candidate B: Hybrid MF + Content

Training HybridMF...
  Train samples: 296,113
  Val samples: 69,055
  Epochs: 3, Batch size: 2048
  Epoch 1/3: Loss=0.8768, RMSE=0.9202, W-RMSE=0.9265 *
  Epoch 2/3: Loss=0.6956, RMSE=0.9235, W-RMSE=0.9289
  Epoch 3/3: Loss=0.5944, RMSE=0.9387, W-RMSE=0.9438
  Early stopping at epoch 3
  Best W-RMSE: 0.9265

Candidate C: Residual Hybrid

Training ResidualHybrid...
  Train samples: 296,113
  Val samples: 69,055
  Epochs: 3, Batch size: 2048
  Epoch 1/3: Loss=0.8943, RMSE=0.9210, W-RMSE=0.9273 *
  Epoch 2/3: Loss=0.7018, RMSE=0.9205, W-RMSE=0.9260 *
  Epoch 3/3: Loss=0.5855, RMSE=0.9413, W-RMSE=0.9452
  Best W-RMSE: 0.9260

Model S

# 10. Full Training
Retrain the best model (optionally with ensemble)

In [ ]:
# %%
"""Full training and ensemble creation."""

class Ensemble:
    """
    Simple weighted ensemble of multiple models.
    """
    
    def __init__(
        self,
        models: List[Tuple[str, nn.Module, float]],  # (name, model, weight)
        feature_builder: FeatureBuilder,
        cfg: Config
    ):
        self.models = models
        self.feature_builder = feature_builder
        self.cfg = cfg
        
        # Normalize weights
        total_weight = sum(w for _, _, w in models)
        self.models = [(n, m, w / total_weight) for n, m, w in models]
        
    @torch.no_grad()
    def predict(
        self,
        user_indices: np.ndarray,
        movie_indices: np.ndarray
    ) -> np.ndarray:
        """Generate predictions using weighted ensemble."""
        all_preds = np.zeros(len(user_indices), dtype=np.float32)
        
        user_tensor = torch.LongTensor(user_indices).to(DEVICE)
        movie_tensor = torch.LongTensor(movie_indices).to(DEVICE)
        content_features = torch.FloatTensor(
            self.feature_builder.get_movie_features(movie_indices)
        ).to(DEVICE)
        
        for name, model, weight in self.models:
            model.eval()
            model.to(DEVICE)
            
            # Check if model uses content features
            try:
                preds = model(user_tensor, movie_tensor, content_features)
            except TypeError:
                # Model doesn't use content features
                preds = model(user_tensor, movie_tensor)
            
            all_preds += weight * preds.cpu().numpy()
        
        if self.cfg.clip_predictions:
            all_preds = np.clip(all_preds, 0.5, 5.0)
        
        if self.cfg.round_to_half:
            all_preds = np.round(all_preds * 2) / 2
            all_preds = np.clip(all_preds, 0.5, 5.0)
        
        return all_preds


def create_ensemble_and_evaluate(
    results: Dict[str, Any],
    val_explicit: pd.DataFrame,
    train_explicit: pd.DataFrame,
    feature_builder: FeatureBuilder,
    cfg: Config
) -> Tuple[Ensemble, float]:
    """
    Create an ensemble from the trained models and evaluate it.
    
    Uses inverse W-RMSE as weights (better models get higher weight).
    """
    # Compute weights based on validation performance
    model_weights = []
    for name, res in results.items():
        # Inverse of W-RMSE as weight
        weight = 1.0 / (res['best_wrmse'] ** 2)
        model_weights.append((name, res['model'], weight))
    
    ensemble = Ensemble(model_weights, feature_builder, cfg)
    
    # Evaluate ensemble on validation
    val_user_idx = val_explicit['user_idx'].values
    val_movie_idx = val_explicit['movie_idx'].values
    val_ratings = val_explicit['rating'].values
    
    ensemble_preds = ensemble.predict(val_user_idx, val_movie_idx)
    
    ensemble_rmse = rmse(val_ratings, ensemble_preds)
    ensemble_wrmse = wrmse(val_ratings, ensemble_preds, val_movie_idx, train_explicit)
    
    print(f"\nEnsemble performance:")
    print(f"  RMSE:  {ensemble_rmse:.4f}")
    print(f"  W-RMSE: {ensemble_wrmse:.4f}")
    
    return ensemble, ensemble_wrmse


# Create ensemble and compare with best single model
print("Creating ensemble from all trained models...")
ensemble, ensemble_wrmse = create_ensemble_and_evaluate(
    selection_results,
    val_explicit,
    train_explicit,
    feature_builder,
    CFG
)

# Decide whether to use ensemble or best single model
best_single_wrmse = selection_results[best_model_name]['best_wrmse']

if ensemble_wrmse < best_single_wrmse:
    print(f"\n*** Using ENSEMBLE (W-RMSE={ensemble_wrmse:.4f} < {best_single_wrmse:.4f}) ***")
    use_ensemble = True
    final_wrmse = ensemble_wrmse
else:
    print(f"\n*** Using single model {best_model_name} (W-RMSE={best_single_wrmse:.4f} <= {ensemble_wrmse:.4f}) ***")
    use_ensemble = False
    final_wrmse = best_single_wrmse

print(f"\nFinal model W-RMSE: {final_wrmse:.4f}")

Creating ensemble from all trained models...

Ensemble performance:
  RMSE:  0.9147
  W-RMSE: 0.9206

*** Using ENSEMBLE (W-RMSE=0.9206 < 0.9260) ***

Final model W-RMSE: 0.9206


# 11. Create Submission
Generate predictions for all rows in ratings_submission.csv

In [ ]:
# %%
"""Generate submission file."""

@torch.no_grad()
def generate_predictions(
    model: nn.Module,
    submission_df: pd.DataFrame,
    id_mapper: IDMapper,
    feature_builder: FeatureBuilder,
    cfg: Config,
    use_content: bool = True,
    batch_size: int = 10000
) -> np.ndarray:
    """
    Generate predictions for submission in batches.
    """
    model.eval()
    model.to(DEVICE)
    
    user_indices = id_mapper.transform_users(submission_df['user_id'].values)
    movie_indices = id_mapper.transform_movies(submission_df['movie_id'].values)
    
    all_preds = []
    
    n_samples = len(user_indices)
    for start_idx in range(0, n_samples, batch_size):
        end_idx = min(start_idx + batch_size, n_samples)
        
        batch_user = torch.LongTensor(user_indices[start_idx:end_idx]).to(DEVICE)
        batch_movie = torch.LongTensor(movie_indices[start_idx:end_idx]).to(DEVICE)
        
        if use_content:
            batch_content = torch.FloatTensor(
                feature_builder.get_movie_features(movie_indices[start_idx:end_idx])
            ).to(DEVICE)
            
            try:
                preds = model(batch_user, batch_movie, batch_content)
            except TypeError:
                preds = model(batch_user, batch_movie)
        else:
            preds = model(batch_user, batch_movie)
        
        all_preds.extend(preds.cpu().numpy())
    
    all_preds = np.array(all_preds, dtype=np.float32)
    
    # Post-processing
    if cfg.clip_predictions:
        all_preds = np.clip(all_preds, 0.5, 5.0)
    
    if cfg.round_to_half:
        all_preds = np.round(all_preds * 2) / 2
        all_preds = np.clip(all_preds, 0.5, 5.0)
    
    return all_preds


def create_submission(
    submission_df: pd.DataFrame,
    predictions: np.ndarray,
    output_path: str = "submission.csv"
) -> pd.DataFrame:
    """
    Create submission file in required format.
    
    Format: id,prediction
    """
    submission = pd.DataFrame({
        'id': submission_df['id'],
        'prediction': predictions
    })
    
    submission.to_csv(output_path, index=False)
    print(f"Submission saved to: {output_path}")
    print(f"  Total predictions: {len(submission):,}")
    print(f"  Prediction range: [{predictions.min():.2f}, {predictions.max():.2f}]")
    print(f"  Prediction mean: {predictions.mean():.4f}")
    
    return submission


# Generate predictions
print("Generating predictions for submission...")

if use_ensemble:
    # Use ensemble
    sub_user_idx = id_mapper.transform_users(submission_df['user_id'].values)
    sub_movie_idx = id_mapper.transform_movies(submission_df['movie_id'].values)
    predictions = ensemble.predict(sub_user_idx, sub_movie_idx)
else:
    # Use best single model
    use_content = best_model_name in ['HybridMF', 'ResidualHybrid']
    predictions = generate_predictions(
        best_model,
        submission_df,
        id_mapper,
        feature_builder,
        CFG,
        use_content=use_content
    )

# Create submission file
submission = create_submission(submission_df, predictions, "submission.csv")

# Show sample
print("\nSubmission sample:")
print(submission.head(10))

Generating predictions for submission...
Submission saved to: submission.csv
  Total predictions: 100,000
  Prediction range: [1.42, 4.92]
  Prediction mean: 3.6546

Submission sample:
         id  prediction
0    0_1188    3.383061
1    5_7163    4.020082
2     8_460    3.442417
3    9_3705    3.704780
4   11_2650    3.328694
5   17_5003    4.250720
6   19_2949    3.985140
7   22_2523    3.364271
8    23_294    3.546657
9  24_48744    4.112661


# 12. Cold-Start Strategy

## What is Cold Start in Recommender Systems?

The **cold-start problem** refers to the challenge of making recommendations when there is insufficient data:

1. **New User Cold Start**: A user who has just joined the platform with no rating history. We cannot compute user embeddings or biases for them.

2. **New Item Cold Start**: A movie that was recently added with no or very few ratings. We cannot compute item embeddings or biases for it.

3. **System Cold Start**: A new system with very little overall data (not applicable here).

### Why is Cold Start Hard?

Collaborative filtering models (like our MF) rely on learning embeddings from user-item interactions. For new users/items:
- No learned embedding exists
- Cannot compute dot products with other embeddings
- The model falls back to global mean, which is suboptimal

### Our Cold-Start Strategy

We implement a **content-based fallback** for cold-start items:
1. Use movie metadata (genres, title, tags) to compute a feature vector
2. Find similar movies with ratings using cosine similarity
3. Use weighted average of similar movies' predictions

For cold-start users:
1. Fall back to movie mean with shrinkage (since we know the movie)
2. Optionally: use popular items as initial recommendations

In [ ]:
# %%
"""Cold-start strategy implementation."""

class ColdStartHandler:
    """
    Handles cold-start predictions for new users and items.
    
    Strategies:
    - New items: Use content-based similarity to known items
    - New users: Fall back to movie statistics
    """
    
    def __init__(
        self,
        feature_builder: FeatureBuilder,
        train_explicit: pd.DataFrame,
        id_mapper: IDMapper,
        n_neighbors: int = 10
    ):
        self.feature_builder = feature_builder
        self.train_explicit = train_explicit
        self.id_mapper = id_mapper
        self.n_neighbors = n_neighbors
        
        # Pre-compute movie statistics
        self.global_mean = train_explicit['rating'].mean()
        
        # Movie means with shrinkage
        shrinkage = 25.0
        movie_stats = train_explicit.groupby('movie_idx')['rating'].agg(['sum', 'count', 'mean'])
        self.movie_means = {}
        self.movie_counts = {}
        for movie_idx, row in movie_stats.iterrows():
            self.movie_means[movie_idx] = (row['sum'] + shrinkage * self.global_mean) / (row['count'] + shrinkage)
            self.movie_counts[movie_idx] = row['count']
        
        # Pre-compute movie feature matrix for similarity
        self.movie_features = feature_builder.all_features
        
        # Normalize features for cosine similarity
        norms = np.linalg.norm(self.movie_features, axis=1, keepdims=True)
        norms[norms == 0] = 1  # Avoid division by zero
        self.movie_features_normed = self.movie_features / norms
        
        # Set of movies with ratings
        self.rated_movies = set(train_explicit['movie_idx'].unique())
        
        # Set of users with ratings
        self.rated_users = set(train_explicit['user_idx'].unique())
        
        print(f"ColdStartHandler initialized:")
        print(f"  Known movies: {len(self.rated_movies):,}")
        print(f"  Known users: {len(self.rated_users):,}")
    
    def is_cold_start_user(self, user_idx: int) -> bool:
        """Check if user is cold-start (no ratings in training)."""
        return user_idx not in self.rated_users or user_idx < 0
    
    def is_cold_start_movie(self, movie_idx: int) -> bool:
        """Check if movie is cold-start (no ratings in training)."""
        return movie_idx not in self.rated_movies or movie_idx < 0
    
    def find_similar_movies(
        self,
        movie_idx: int,
        n_neighbors: int = None
    ) -> List[Tuple[int, float]]:
        """
        Find similar movies using content features.
        
        Returns:
            List of (movie_idx, similarity) tuples
        """
        if n_neighbors is None:
            n_neighbors = self.n_neighbors
        
        if movie_idx < 0 or movie_idx >= len(self.movie_features_normed):
            return []
        
        # Compute cosine similarity
        query_features = self.movie_features_normed[movie_idx]
        similarities = self.movie_features_normed @ query_features
        
        # Get top neighbors (excluding self and movies without ratings)
        neighbor_scores = []
        for idx in np.argsort(similarities)[::-1]:
            if idx != movie_idx and idx in self.rated_movies:
                neighbor_scores.append((idx, similarities[idx]))
                if len(neighbor_scores) >= n_neighbors:
                    break
        
        return neighbor_scores
    
    def predict_cold_start_item(
        self,
        movie_idx: int,
        user_idx: int = None
    ) -> float:
        """
        Predict rating for a cold-start movie.
        
        Uses weighted average of similar movies' mean ratings.
        """
        similar_movies = self.find_similar_movies(movie_idx)
        
        if not similar_movies:
            return self.global_mean
        
        # Weighted average of similar movies' means
        weighted_sum = 0.0
        weight_total = 0.0
        
        for sim_movie_idx, similarity in similar_movies:
            if similarity > 0:
                movie_mean = self.movie_means.get(sim_movie_idx, self.global_mean)
                weighted_sum += similarity * movie_mean
                weight_total += similarity
        
        if weight_total > 0:
            return weighted_sum / weight_total
        return self.global_mean
    
    def predict_cold_start_user(
        self,
        movie_idx: int
    ) -> float:
        """
        Predict rating for a cold-start user.
        
        Falls back to movie mean (with shrinkage).
        """
        if movie_idx in self.movie_means:
            return self.movie_means[movie_idx]
        return self.global_mean
    
    def predict(
        self,
        user_idx: int,
        movie_idx: int
    ) -> Tuple[float, str]:
        """
        Get cold-start prediction and method used.
        
        Returns:
            (prediction, method_name)
        """
        is_cold_user = self.is_cold_start_user(user_idx)
        is_cold_movie = self.is_cold_start_movie(movie_idx)
        
        if is_cold_movie and is_cold_user:
            # Both cold start - use content-based for movie
            pred = self.predict_cold_start_item(movie_idx)
            return pred, "cold_both_content"
        elif is_cold_movie:
            # Only movie is cold - use content similarity
            pred = self.predict_cold_start_item(movie_idx, user_idx)
            return pred, "cold_movie_content"
        elif is_cold_user:
            # Only user is cold - use movie mean
            pred = self.predict_cold_start_user(movie_idx)
            return pred, "cold_user_movie_mean"
        else:
            # Not cold start - shouldn't call this
            return None, "not_cold_start"

print("ColdStartHandler class defined.")

ColdStartHandler class defined.


In [ ]:
# %%
"""Evaluate cold-start strategy against naive baselines."""

def evaluate_cold_start_strategy(
    train_explicit: pd.DataFrame,
    val_explicit: pd.DataFrame,
    feature_builder: FeatureBuilder,
    id_mapper: IDMapper,
    global_mean: float
):
    """
    Evaluate cold-start strategy by simulating cold-start scenarios.
    
    Creates artificial cold-start by removing some movies from training
    and measuring performance on their validation ratings.
    """
    print("\n" + "="*60)
    print("Cold-Start Strategy Evaluation")
    print("="*60)
    
    # Initialize cold-start handler with full training data
    cold_handler = ColdStartHandler(
        feature_builder=feature_builder,
        train_explicit=train_explicit,
        id_mapper=id_mapper,
        n_neighbors=10
    )
    
    # Find movies that appear in validation but have few ratings in training
    val_movies = set(val_explicit['movie_idx'].unique())
    train_movie_counts = train_explicit.groupby('movie_idx').size()
    
    # Simulate cold-start: movies with < 5 ratings in training
    cold_threshold = 5
    cold_movies = set()
    for movie_idx in val_movies:
        count = train_movie_counts.get(movie_idx, 0)
        if count < cold_threshold:
            cold_movies.add(movie_idx)
    
    print(f"\nSimulated cold-start movies (< {cold_threshold} training ratings): {len(cold_movies)}")
    
    # Get validation data for cold-start movies
    val_cold = val_explicit[val_explicit['movie_idx'].isin(cold_movies)].copy()
    
    if len(val_cold) == 0:
        print("No cold-start validation samples found. Trying lower threshold...")
        cold_threshold = 20
        cold_movies = set()
        for movie_idx in val_movies:
            count = train_movie_counts.get(movie_idx, 0)
            if count < cold_threshold:
                cold_movies.add(movie_idx)
        val_cold = val_explicit[val_explicit['movie_idx'].isin(cold_movies)].copy()
        print(f"With threshold < {cold_threshold}: {len(val_cold)} samples")
    
    if len(val_cold) == 0:
        print("Skipping cold-start evaluation - insufficient data")
        return
    
    # Prepare cold-start evaluation (pretend these movies have no training data)
    # Create a modified cold-start handler that doesn't know about these movies
    train_filtered = train_explicit[~train_explicit['movie_idx'].isin(cold_movies)]
    
    cold_handler_eval = ColdStartHandler(
        feature_builder=feature_builder,
        train_explicit=train_filtered,
        id_mapper=id_mapper,
        n_neighbors=10
    )
    
    # Baseline 1: Global mean (naive)
    preds_global = np.full(len(val_cold), global_mean, dtype=np.float32)
    rmse_global = rmse(val_cold['rating'].values, preds_global)
    
    # Baseline 2: Random prediction
    np.random.seed(42)
    preds_random = np.random.uniform(0.5, 5.0, size=len(val_cold)).astype(np.float32)
    rmse_random = rmse(val_cold['rating'].values, preds_random)
    
    # Our content-based cold-start strategy
    preds_content = []
    for _, row in val_cold.iterrows():
        pred, method = cold_handler_eval.predict(int(row['user_idx']), int(row['movie_idx']))
        preds_content.append(pred if pred is not None else global_mean)
    preds_content = np.array(preds_content, dtype=np.float32)
    rmse_content = rmse(val_cold['rating'].values, preds_content)
    
    print(f"\n--- Cold-Start Evaluation Results ({len(val_cold)} samples) ---")
    print(f"  Random baseline:        RMSE = {rmse_random:.4f}")
    print(f"  Global mean baseline:   RMSE = {rmse_global:.4f}")
    print(f"  Content-based strategy: RMSE = {rmse_content:.4f}")
    
    improvement_vs_global = (rmse_global - rmse_content) / rmse_global * 100
    improvement_vs_random = (rmse_random - rmse_content) / rmse_random * 100
    
    print(f"\n  Improvement vs global mean: {improvement_vs_global:+.2f}%")
    print(f"  Improvement vs random:      {improvement_vs_random:+.2f}%")
    
    if rmse_content < rmse_global:
        print("\n  ✓ Content-based cold-start beats naive baselines!")
    else:
        print("\n  Note: Content similarity may not help for this sample. Consider:")
        print("    - Using more neighbors")
        print("    - Adding more content features")
        print("    - Using hybrid with popularity bias")
    
    return {
        'n_cold_movies': len(cold_movies),
        'n_samples': len(val_cold),
        'rmse_random': rmse_random,
        'rmse_global': rmse_global,
        'rmse_content': rmse_content
    }

# Run cold-start evaluation
cold_start_results = evaluate_cold_start_strategy(
    train_explicit,
    val_explicit,
    feature_builder,
    id_mapper,
    GLOBAL_MEAN
)


Cold-Start Strategy Evaluation
ColdStartHandler initialized:
  Known movies: 2,000
  Known users: 87,977

Simulated cold-start movies (< 5 training ratings): 0
No cold-start validation samples found. Trying lower threshold...
With threshold < 20: 0 samples
Skipping cold-start evaluation - insufficient data


In [ ]:
# %%
"""Integrated prediction function that handles cold-start cases."""

class HybridPredictor:
    """
    Full predictor that combines learned models with cold-start handling.
    """
    
    def __init__(
        self,
        model: nn.Module,
        ensemble: Ensemble,
        use_ensemble: bool,
        cold_handler: ColdStartHandler,
        feature_builder: FeatureBuilder,
        id_mapper: IDMapper,
        cfg: Config,
        use_content: bool = True
    ):
        self.model = model
        self.ensemble = ensemble
        self.use_ensemble = use_ensemble
        self.cold_handler = cold_handler
        self.feature_builder = feature_builder
        self.id_mapper = id_mapper
        self.cfg = cfg
        self.use_content = use_content
        
    @torch.no_grad()
    def predict_batch(
        self,
        user_ids: np.ndarray,
        movie_ids: np.ndarray
    ) -> np.ndarray:
        """
        Generate predictions handling cold-start cases.
        """
        user_indices = self.id_mapper.transform_users(user_ids)
        movie_indices = self.id_mapper.transform_movies(movie_ids)
        
        predictions = np.zeros(len(user_ids), dtype=np.float32)
        cold_start_mask = np.zeros(len(user_ids), dtype=bool)
        
        # Identify cold-start cases
        for i, (u_idx, m_idx) in enumerate(zip(user_indices, movie_indices)):
            if self.cold_handler.is_cold_start_user(u_idx) or self.cold_handler.is_cold_start_movie(m_idx):
                cold_start_mask[i] = True
                pred, _ = self.cold_handler.predict(u_idx, m_idx)
                predictions[i] = pred if pred is not None else self.cold_handler.global_mean
        
        # Handle non-cold-start cases with the model
        non_cold_indices = np.where(~cold_start_mask)[0]
        
        if len(non_cold_indices) > 0:
            if self.use_ensemble:
                model_preds = self.ensemble.predict(
                    user_indices[non_cold_indices],
                    movie_indices[non_cold_indices]
                )
            else:
                self.model.eval()
                self.model.to(DEVICE)
                
                batch_user = torch.LongTensor(user_indices[non_cold_indices]).to(DEVICE)
                batch_movie = torch.LongTensor(movie_indices[non_cold_indices]).to(DEVICE)
                
                if self.use_content:
                    batch_content = torch.FloatTensor(
                        self.feature_builder.get_movie_features(movie_indices[non_cold_indices])
                    ).to(DEVICE)
                    try:
                        model_preds = self.model(batch_user, batch_movie, batch_content).cpu().numpy()
                    except TypeError:
                        model_preds = self.model(batch_user, batch_movie).cpu().numpy()
                else:
                    model_preds = self.model(batch_user, batch_movie).cpu().numpy()
            
            predictions[non_cold_indices] = model_preds
        
        # Post-processing
        if self.cfg.clip_predictions:
            predictions = np.clip(predictions, 0.5, 5.0)
        
        if self.cfg.round_to_half:
            predictions = np.round(predictions * 2) / 2
            predictions = np.clip(predictions, 0.5, 5.0)
        
        n_cold = cold_start_mask.sum()
        if n_cold > 0:
            print(f"  Handled {n_cold:,} cold-start predictions ({100*n_cold/len(user_ids):.2f}%)")
        
        return predictions


# Create the full hybrid predictor
cold_handler = ColdStartHandler(
    feature_builder=feature_builder,
    train_explicit=train_explicit,
    id_mapper=id_mapper,
    n_neighbors=10
)

hybrid_predictor = HybridPredictor(
    model=best_model,
    ensemble=ensemble,
    use_ensemble=use_ensemble,
    cold_handler=cold_handler,
    feature_builder=feature_builder,
    id_mapper=id_mapper,
    cfg=CFG,
    use_content=best_model_name in ['HybridMF', 'ResidualHybrid']
)

print("HybridPredictor with cold-start handling created.")

ColdStartHandler initialized:
  Known movies: 2,000
  Known users: 87,977
HybridPredictor with cold-start handling created.


In [ ]:
# %%
"""Final submission with cold-start handling."""

print("\n" + "="*60)
print("Generating Final Submission with Cold-Start Handling")
print("="*60)

# Generate predictions with cold-start handling
final_predictions = hybrid_predictor.predict_batch(
    submission_df['user_id'].values,
    submission_df['movie_id'].values
)

# Create final submission
final_submission = create_submission(
    submission_df,
    final_predictions,
    "submission.csv"
)

print("\n--- Final Summary ---")
print(f"  Configuration: {CFG.name}")
print(f"  Best model: {best_model_name}")
print(f"  Using ensemble: {use_ensemble}")
print(f"  Validation W-RMSE: {final_wrmse:.4f}")
print(f"  Total predictions: {len(final_submission):,}")

# Show distribution of predictions
print("\n--- Prediction Distribution ---")
print(f"  Min:    {final_predictions.min():.2f}")
print(f"  Max:    {final_predictions.max():.2f}")
print(f"  Mean:   {final_predictions.mean():.4f}")
print(f"  Median: {np.median(final_predictions):.4f}")
print(f"  Std:    {final_predictions.std():.4f}")

# Show final submission sample
print("\n--- Final Submission Sample ---")
print(final_submission.head(10))


Generating Final Submission with Cold-Start Handling
  Handled 12,023 cold-start predictions (12.02%)
Submission saved to: submission.csv
  Total predictions: 100,000
  Prediction range: [1.42, 4.92]
  Prediction mean: 3.6561

--- Final Summary ---
  Configuration: CPU_DEBUG
  Best model: ResidualHybrid
  Using ensemble: True
  Validation W-RMSE: 0.9206
  Total predictions: 100,000

--- Prediction Distribution ---
  Min:    1.42
  Max:    4.92
  Mean:   3.6561
  Median: 3.7224
  Std:    0.4137

--- Final Submission Sample ---
         id  prediction
0    0_1188    3.383061
1    5_7163    4.085719
2     8_460    3.470111
3    9_3705    3.704780
4   11_2650    3.328694
5   17_5003    4.250720
6   19_2949    3.985140
7   22_2523    3.364271
8    23_294    3.546657
9  24_48744    4.112661


# Summary and Next Steps

## What We Built

1. **Data Pipeline**: Efficient loading with proper dtypes, ID mappings, and feature extraction
2. **Feature Engineering**:
   - Genre multi-hot encoding
   - Title TF-IDF with SVD dimensionality reduction
   - Tag aggregation with TF-IDF and SVD
3. **Candidate Models**:
   - **BiasedMF**: Classic matrix factorization with user/item biases
   - **HybridMF**: MF + content features with MLP fusion
   - **ResidualHybrid**: MF with content-based residual correction
4. **Model Selection**: Automatic selection based on W-RMSE
5. **Ensemble**: Weighted combination of all models
6. **Cold-Start Handling**: Content-based similarity fallback

## Key Metrics

- Weighted RMSE as primary metric (down-weights popular movies)
- Standard RMSE for debugging
- Comparison against simple baselines (global mean, movie mean, bias model)

## To Improve Further

1. **For GPU_FULL mode**:
   - Increase embedding dimensions and hidden layers
   - Train for more epochs with larger batch sizes
   - Enable mixed precision training
   
2. **Additional ideas**:
   - Use implicit feedback more extensively (BPR loss, exposure modeling)
   - Add user-level content features (aggregated tags, genre preferences)
   - Try temporal features (if timestamps available)
   - Hyperparameter tuning with Optuna
   - Cross-validation for more robust model selection

In [ ]:
# %%
"""Cleanup and memory report."""

# Clean up GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Memory usage
gc.collect()

print("\n" + "="*60)
print("Pipeline Complete!")
print("="*60)
print(f"\nSubmission file saved: submission.csv")
print(f"Ready for Kaggle submission.")
print("\nTo switch to GPU_FULL mode for production:")
print("  1. Change 'CFG = CPU_DEBUG' to 'CFG = GPU_FULL'")
print("  2. Re-run all cells")
print("  3. Expected training time: ~30-60 minutes on GPU")